## **Projeto de Aprendizagem Computacional 2**

### Grupo 1: Carlos Pereira, João Tomás Monteiro Dias: 202404711, Maria Santos: 202404276 (pedir ups)

**Objetivo**: treinar um modelo capaz de detectar sopros cardíacos a partir de fonocardiogramas.

**Modelos escolhidos**: CNN2D e RNN 


### **CNN2D** : Arquitetura

In [ ]:
from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)

CLASSES = ["Present", "Unknown", "Absent"]  # 0, 1, 2
METRIC_W = np.array([5.0, 3.0, 1.0])


# =====================================================================
# 1. ARQUITETURA DA CNN2D
# =====================================================================
class MurmurCNN:
    @staticmethod
    def build(frames=153, n_mels=40, n_classes=3):
        inp = keras.Input(shape=(frames, n_mels, 1))

        # Bloco 1: 16 filtros
        x = layers.Conv2D(16, 3, padding="same", activation="relu")(inp)
        x = layers.BatchNormalization(momentum=0.9)(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(0.25)(x)

        # Bloco 2: 32 filtros
        x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization(momentum=0.9)(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(0.25)(x)

        # Bloco 3: 64 filtros
        x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization(momentum=0.9)(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(0.25)(x)

        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(64, activation="relu")(x)
        x = layers.Dropout(0.5)(x)
        out = layers.Dense(n_classes, activation="softmax")(x)

        return keras.Model(inp, out, name="Custom_CNN2D")


## Pré Processamento 

In [ ]:
import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from scipy.io import wavfile
from scipy.signal import butter, sosfiltfilt, resample_poly, spectrogram
from sklearn.model_selection import train_test_split

CLASSES = ["Present", "Unknown", "Absent"]
CLASS2IDX = {c: i for i, c in enumerate(CLASSES)}

@dataclass
class Config:
    data_dir: Path = Path("../data/circor")
    fs_target: int = 1000   # Hz (Nyquist = 500 Hz)
    band: tuple = (25, 400)        # passa-banda (Hz)
    win_sec: float = 5.0           # tamanho da janela (s)
    hop_sec: float = 2.5           # passo entre janelas (s)
    # --- log-mel / CNN 2D ---
    n_mels: int = 40
    frame_len: int = 128           # amostras (128 ms a 1 kHz)
    frame_step: int = 32           # amostras (32 ms)
    # --- RNN ---
    n_mfcc: int = 13
    rnn_frame_step: int = 64       # passo maior => sequências mais curtas (~77)
    seed: int = 42

    @property
    def wav_dir(self):
        return self.data_dir / "training_data"

    @property
    def csv_path(self):
        return self.data_dir / "training_data.csv"

CFG = Config()
print("CSV existe:", CFG.csv_path.exists(), "| pasta wav existe:", CFG.wav_dir.exists())

## Metadados e split por paciente
O split é feito **por paciente** (todas as gravações de um paciente ficam no mesmo subconjunto) e **estratificado** pela classe `Murmur`. O resultado é guardado em `splits.json`.

In [ ]:
def load_metadata(cfg: Config = CFG) -> pd.DataFrame:
    """Carrega o arquivo CSV com os metadados dos áudios."""
    df = pd.read_csv(cfg.csv_path)[["Patient ID", "Murmur"]].copy()
    df.columns = ["pid", "murmur"]
    df["pid"] = df["pid"].astype(str)
    df["y"] = df["murmur"].map(CLASS2IDX)
    assert df["y"].notna().all(), "Valores de Murmur inesperados"
    return df

def make_splits(meta: pd.DataFrame, val_size=0.15, test_size=0.15, seed=42):
    """Split estratificado POR PACIENTE (uma linha do meta = um paciente)."""
    pids, ys = meta["pid"].values, meta["y"].values
    trval, test = train_test_split(pids, test_size=test_size, stratify=ys, random_state=seed)
    y_traval = meta.set_index("pid").loc[trval, "y"].values
    train, val = train_test_split(trval, test_size=val_size / (1 - test_size), stratify=y_traval, random_state=seed)
    return {"train": list(train), "val": list(val), "test": list(test)}

def get_splits(meta, path="splits.json", seed=42):
    """Carrega o split guardado ou cria-o (todos no grupo devem usar o mesmo)."""
    path = Path(path)
    if path.exists():
        return json.loads(path.read_text())
    splits = make_splits(meta, seed=seed)
    path.write_text(json.dumps(splits))
    return splits

In [ ]:
meta = load_metadata()
splits = get_splits(meta)

print(meta["murmur"].value_counts(normalize=True).round(3), "\n")
for k, v in splits.items():
    print(k, len(v), meta[meta.pid.isin(v)].murmur.value_counts().to_dict())

# Garantia: sem pacientes repetidos entre subconjuntos
assert not (set(splits["train"]) & set(splits["val"]))
assert not (set(splits["train"]) & set(splits["test"]))
assert not (set(splits["val"]) & set(splits["test"]))

## Etapa 1 — Pré-processamento comum
- **Reamostragem** 4000 → 1000 Hz: a banda útil está abaixo de 500 Hz (Nyquist), e assim há 4× menos amostras.
- **Passa-banda Butterworth 25–400 Hz** com `sosfiltfilt` (fase zero): remove deriva de baixa frequência e ruído de alta frequência sem deslocar S1/S2 no tempo.
- **Z-score** por gravação: remove diferenças de amplitude/ganho entre gravações.
- **Janelas** de 5 s (hop 2.5 s): entrada de tamanho fixo + mais exemplos. Cada janela herda o rótulo do paciente.

In [ ]:
def load_recording(path, cfg: Config = CFG) -> np.ndarray:
    """Lê .wav, reamostra para fs_target, filtra e normaliza (z-score)."""
    fs, x = wavfile.read(path)
    x = x.astype(np.float32)
    x = x / (np.abs(x).max() + 1e-8) # normaliza para [-1, 1]
    g = np.gcd(int(fs), cfg.fs_target)
    x = resample_poly(x, cfg.fs_target // g, int(fs // g)).astype(np.float32)
    sos = butter(4, cfg.band, btype="bandpass", fs=cfg.fs_target, output="sos")
    x = sosfiltfilt(sos, x).astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-8) # normaliza para z-score

def to_windows(x: np.ndarray, cfg: Config = CFG) -> np.ndarray:
    """Janelas de win_sec com passo hop_sec (faz padding se for mais curto)."""
    w, h = int(cfg.win_sec * cfg.fs_target), int(cfg.hop_sec * cfg.fs_target)
    if len(x) < w:
        x = np.pad(x, (0, w - len(x)))
    return np.stack([x[s:s + w] for s in range(0, len(x) - w + 1, h)])

def build_waveform_dataset(pids, meta: pd.DataFrame, cfg: Config = CFG):
    """Para cada paciente, processa TODAS as gravações (<pid>_<local>[_n].wav).

    Devolve:
      waves  (N, amostras) float32
      info   DataFrame com uma linha por janela: pid, loc, rec, y
    """
    ymap = meta.set_index("pid")["y"].to_dict()
    waves, rows = [], []
    for pid in pids: 
        for wav in sorted(cfg.wav_dir.glob(f"{pid}_*.wav")):
            loc = wav.stem.split("_")[1]    # AV, PV, TV, MV, Phc...
            win = to_windows(load_recording(wav, cfg), cfg)
            waves.append(win)
            rows += [dict(pid=pid, loc=loc, rec=wav.stem, y=ymap[pid])] * len(win)
    return np.concatenate(waves), pd.DataFrame(rows)

# Etapa 2 - Features específicas para cada modelo
| Função | Saída | Modelo |
|---|---|---|
| `features_cnn2d` | `(N, frames, 40, 1)` | CNN 2D (log-mel como "imagem") |
| `features_rnn` | `(N, ~77, 39)` | RNN (cada frame é um passo temporal) |

- **STFT → mel → log**: mostra a energia no tempo e na frequência; o log comprime a gama dinâmica (sopros têm muito menos energia que S1/S2).
- **MFCC** (para a RNN): DCT do log-mel, decorrelaciona as bandas e comprime a informação; os **deltas** acrescentam a variação temporal.
- Para a RNN usa-se um passo de frame maior (64) para obter sequências mais curtas.

In [ ]:
def _log_mel_tf(w, cfg: Config, frame_step: int):
    n_bins = cfg.frame_len // 2 + 1
    stft = tf.signal.stft(w, cfg.frame_len, frame_step, fft_length=cfg.frame_len)
    mel_w = tf.signal.linear_to_mel_weight_matrix(cfg.n_mels, n_bins, cfg.fs_target, cfg.band[0], cfg.band[1])
    return tf.math.log(tf.tensordot(tf.abs(stft) ** 2, mel_w, 1) + 1e-6)

def features_cnn2d(waves, cfg: Config = CFG) -> np.ndarray:
    """CNN 2D: log-mel -> (N, frames, n_mels, 1), z-score por janela."""
    def f(w):
        lm = _log_mel_tf(w, cfg, cfg.frame_step).numpy()
        return _zscore(lm, (1, 2))[..., None].astype(np.float32)
    return _batched(f, waves)

def features_rnn(waves, kind="mfcc", delta=True, cfg: Config = CFG) -> np.ndarray:
    """RNN: sequência (N, passos, features).

    kind="mfcc"   : n_mfcc coeficientes (+ delta e delta-delta se deltas=True)
    kind="logmel" : n_mels bandas log-mel
    """
    step = cfg.rnn_frame_step

    def f(w):
        lm = _log_mel_tf(w, cfg, step)
        if kind == "logmel":
            return _zscore(lm.numpy(), (1,)).astype(np.float32)
        mf = tf.signal.mfccs_from_log_mel_spectrograms(lm)[..., :cfg.n_mfcc].numpy()
        feats = [mf]
        if delta:
            d1 = np.gradient(mf, axis=1)
            feats += [d1, np.gradient(d1, axis=1)]
        return _zscore(np.concatenate(feats, axis=-1), (1,)).astype(np.float32)
    return _batched(f, waves)

## 4. Gerar e guardar as features para todos os splits

Cria `cache/{train,val,test}.npz` (com `cnn2d`, `rnn`, `y`) e `cache/{split}_info.csv` (com `pid`, `loc`, `rec`, `y` por janela — necessário para agregar por paciente).

In [ ]:
out = Path("cache")
out.mkdir(exist_ok=True)

for name, pids in splits.items():
    waves, info = build_waveform_dataset(pids, meta)
    info.to_csv(out / f"{name}_info.csv", index=False)
    np.savez_compressed(
        out / f"{name}.npz",
        y=info["y"].values,
        cnn2d=features_cnn2d(waves),
        rnn=features_rnn(waves, "mfcc"),
    )
    print(f"{name}: {len(pids)} pacientes, {len(waves)} janelas")

## Estratégia de Treino

Foi desenhada para resolver os dois maiores desafios deste problema (a deteção de sopros cardíacos da competição CirCor DigiScope): o desbalanceamento severo das classes e o desfasamento entre a unidade de entrada do modelo (janela de áudio) e a unidade de diagnóstico clínico (paciente).

- A CNN 2D aprende características locais (tempo-frequência) analisando cada janela de tempo isolada.  

- Agregação por Paciente (to_patient): agrupa todas as janelas que partilham o mesmo pid e calcula a média das probabilidades previstas para esse doente:

    $$\bar{P}(\text{classe} \mid \text{paciente}) = \frac{1}{K} \sum_{k=1}^K P(\text{classe} \mid \text{janela}_k)$$

- Compensação de Classes Desbalanceadas no Treino (class_weight):   Absent: $\approx 73.8\%$   Present: $\approx 19.0\%$   Unknown: $\approx 7.2\%$. 
Para contrariar o desbalancemanto, calculamos os pesos de classe inversamente proporcionais às contagens do conjunto de treino:
    $$\text{weight}_i = \frac{N_{\text{total}}}{3 \times N_i}$$

- **Otimização Orientada à Métrica da Competição (Weighted Accuracy - WA)** : A métrica oficial da competição PhysioNet atribui recompensas desiguais aos acertos com base no vetor de pesos METRIC_W = [5.0, 3.0, 1.0]:  
 -  Present: peso $5.0$ |  Unknown: peso $3.0$ |  Absent: peso $1.0$

Esta otimização é implementada pelas:

- Regra de Decisão do Doente: mean, ou seja as as probabilidades médias eram ponderadas por este vetor (score = pr * METRIC_W) antes de passarem pelo argmax;

- Callback Customizado (PatientWA): no final de cada epoch, a função on_epoch_end corre a validação completa, agrega por paciente e calcula o WA real da validação (val_wa), permitindo monitorizar a weighted accuracy e não apenas a accuracy por janela;

**Gestão Dinâmica da Convergência (Callbacks)**: O treino utiliza dois mecanismos automáticos para evitar overfitting e garantir que a rede converge de forma estável:   
- ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3): Se o erro de validação (val_loss) não diminuir durante 3 épocas consecutivas, a taxa de aprendizagem (learning rate) é reduzida para metade (passando de $0.001$ para $0.0005$, depois $0.00025$, etc.), permitindo que a descida do gradiente refine os pesos.   
- EarlyStopping(monitor="val_wa", mode="max", patience=8, restore_best_weights=True): vigia diretamente o val_wa ao nível do paciente. Se o modelo passar 8 épocas sem conseguir superar o seu recorde de WA, o treino é interrompido e os pesos da melhor época são automaticamente restaurados para o ficheiro final.

**Estas classes estão definidas no ficheiro 'train_strategy.py'**

------------------------------------------------------------------------------------------------------------------------------------------

## CNN2D: Resultados com a estratégia inicial


In [ ]:

CACHE = Path("../dados")
d_tr = np.load(CACHE / "train.npz")
info_tr = pd.read_csv(CACHE / "train_info.csv", dtype={"pid": str})
Xtr, ytr, ptr = d_tr["cnn2d"], d_tr["y"], info_tr["pid"].values

d_va = np.load(CACHE / "val.npz")
info_va = pd.read_csv(CACHE / "val_info.csv", dtype={"pid": str})
Xva, yva, pva = d_va["cnn2d"], d_va["y"], info_va["pid"].values


In [ ]:
from train_strategy import MurmurTrainer, PatientAggregator
model_baseline = MurmurCNN.build(frames=Xtr.shape[1], n_mels=Xtr.shape[2])
trainer_base = MurmurTrainer(model_baseline, weight_mode="standard")
agg_base = PatientAggregator(strategy="mean", decision="weighted")

trainer_base.train(Xtr, ytr, Xva, yva, pva, aggregator=agg_base, epochs=30)

probs_base = model_baseline.predict(Xva, batch_size=128, verbose=0)
yt_b, yp_b, _ = agg_base.aggregate(probs_base, yva, pva)

# Imprime imediatamente o WA_murmur e a matriz de confusão da Baseline
wa_base, _ = trainer_base.evaluate_and_report(
    Xva, yva, pva,
    aggregator=agg_base,
    split_name="VALIDAÇÃO (BASELINE)",
    optimize_thresholds=False
)


- Treino com 1 split: 

**wa_murmur: 0.6778**

| | Pred Present | Pred Unknown | Pred Absent | Total | Recall |
|---|---|---|---|---|---|
| **True Present** | 27 | 0 | 0 | 27 | 1,00 |
| **True Unknown** | 4 | 6 | 0 | 10 | 0,60 |
| **True Absent** | 62 | 13 | 30 | 105 | 0,29 |



## Alterações na estratégia de treino e previsão

**Suavização dos pesos das classes no treino (Smoothed Class Weights) com a raiz quadrada**: Substituir os pesos agressivos pela raiz quadrada da frequência inversa das classes (normalizada para média $1.0$), permitindo que a rede calibre probabilidades mais realistas sem ficar demasiado polarizada, mantendo a ponderação 5:3:1 apenas na avaliação final.

**Agregação dos pacientes: Top-K = 5** : em vez de calcularmos a média de todas as janelas temporais de cada paciente, fazemos a média das 5 janelas do doente que têm probabilidade mais elevada de sopro (Present). Evita que um sopro nítido em poucas gravações/janelas seja diluído por várias janelas sem sinal audível.

**Decisão `unweighted` em vez de `weighted`**: o argmax passa a ser aplicado diretamente às probabilidades do doente, sem as multiplicar por [5, 3, 1]. O WA continua a ser calculado com os pesos 5:3:1 na avaliação.


**Estas alterações encontram-se no ficheiro train_strategy.py e podem ser passadas como argumento na chamada do modelo CNN2D.**




In [ ]:
model_improved = MurmurCNN.build(frames=Xtr.shape[1], n_mels=Xtr.shape[2])
trainer_imp = MurmurTrainer(model_improved, weight_mode="smoothed")
agg_imp = PatientAggregator(strategy="topk", k=5, decision="unweighted")

trainer_imp.train(Xtr, ytr, Xva, yva, pva, aggregator=agg_imp, epochs=30)

probs_imp = model_improved.predict(Xva, batch_size=128, verbose=0)
yt_i, yp_i, _ = agg_imp.aggregate(probs_imp, yva, pva)

wa_imp, _ = trainer_imp.evaluate_and_report(
    Xva, yva, pva,
    aggregator=agg_imp,
    split_name="VALIDAÇÃO (IMPROVED)",
    optimize_thresholds=False
)




## Resultados obtidos com as alterações:

- treino com 1 split:

**WA_murmur: 0.8296**

| | Pred Present | Pred Unknown | Pred Absent | Total | Recall |
|---|---|---|---|---|---|
| **True Present** | 27 | 0 | 0 | 27 | 1,00 |
| **True Unknown** | 5 | 1 | 4 | 10 | 0,10 |
| **True Absent** | 18 | 1 | 86 | 105 | 0,82 |

**Validação cruzada (5 folds, Top-K = 5, decisão sem pesos): métricas por doente**

| Fold | WA_murmur | Accuracy | Balanced acc. | Macro-F1 |
|---|---|---|---|---|
| 0 | 0,7708 | 0,8302 | 0,5800 | 0,5436 |
| 1 | 0,7500 | 0,8125 | 0,5911 | 0,5770 |
| 2 | 0,7876 | 0,7062 | 0,6199 | 0,5447 |
| 3 | 0,7697 | 0,7750 | 0,5990 | 0,5546 |
| 4 | 0,7994 | 0,8385 | 0,6453 | 0,6399 |
| **Média ± dp** | **0,7755 ± 0,0188** | 0,7925 ± 0,0540 | 0,6071 ± 0,0259 | 0,5719 ± 0,0403 |
| **Agregado (5 folds)** | 0,7756 | 0,7925 | 0,6074 | 0,5740 |

**Métricas por classe (doentes dos 5 folds agregados)**

| Classe | Precisão | Recall | Especificidade | F1 | Nº doentes |
|---|---|---|---|---|---|
| Present | 0,593 | 0,882 | 0,858 | 0,709 | 152 |
| Unknown | 0,250 | 0,103 | 0,976 | 0,146 | 58 |
| Absent | 0,898 | 0,837 | 0,733 | 0,867 | 590 |

#### **O k-fold cross-validation está definido no ficheiro kfold_cv.py**


![curvas k-fold](figs/kfold_slide.png)

## **Teste:**

### Utilizamos um ensemble dos 5 modelos treinados na validação cruzada

Para a avaliação final usámos o conjunto de teste (`test.npz`), que nunca foi usado no treino, na validação interna (early stopping) nem na escolha de hiperparâmetros. Os modelos avaliados são os 5 obtidos na validação cruzada estratificada por doente (5 folds, `StratifiedGroupKFold`), todos treinados com a mesma configuração (pesos de classe `[smoothed]`, seed 42).

**Procedimento:**

1. Cada um dos 5 modelos (um por fold) prevê as probabilidades de cada janela do conjunto de teste.
2. As probabilidades são **médias entre os 5 modelos**, janela a janela (ensemble por média de probabilidades).
3. As previsões das janelas são agregadas por doente com **Top-K** (K = 5), e a classe final é o argmax das probabilidades agregadas.
4. Calculamos o WA_murmur (pesos 5:3:1 para Present:Unknown:Absent) e a matriz de confusão ao nível do doente.



In [ ]:
MODEL_PATHS = [
    f"cv/fold{f}/model.keras" for f in range(5)
]
TEST_NPZ = "dados/test.npz"

# Carregar dados de teste e modelos
data = np.load(TEST_NPZ, allow_pickle=True)
X_test = data["cnn2d"]
if X_test.ndim == 3:
    X_test = X_test[..., np.newaxis]
y_test = data["y"]

pids_test = pd.read_csv("dados/test_info.csv")["pid"].values

models = [keras.models.load_model(p) for p in MODEL_PATHS]


#Prever janelas (média dos 5 folds) e agregar por paciente
probs_win = np.mean([m.predict(X_test, batch_size=256, verbose=0) for m in models], axis=0)

agg = PatientAggregator(strategy="topk", k=5, decision="unweighted")
yt, yp, _ = agg.aggregate(probs_win, y_test, pids_test)

#Calcular WA e Matriz de Confusão 
w = METRIC_W[yt]
wa = float(np.sum(w * (yt == yp)) / np.sum(w))

cm = np.zeros((3, 3), dtype=int)
for t, p in zip(yt, yp):
    cm[p, t] += 1

print(f"WA_murmur (Teste - Ensemble 5 Folds): {wa:.4f}\n")
print(pd.DataFrame(cm, index=[f"Pred {c}" for c in CLASSES], columns=[f"True {c}" for c in CLASSES]))

**Resultados CNN2D**
- Pesos suavizados;
- Agregação Top K=5;
- Decisão não ponderada (unweighted).


**WA_murmur (Teste - Ensemble 5 Folds): 0.7852**

| | Pred Present | Pred Unknown | Pred Absent | Total | Recall |
|---|---|---|---|---|---|
| **True Present** | 22 | 0 | 5 | 27 | 0,81 |
| **True Unknown** | 3 | 1 | 6 | 10 | 0,10 |
| **True Absent** | 6 | 0 | 99 | 105 | 0,94 |


## **Tabela resumo das alterações**

| | Baseline | Final |
|---|---|---|
| Pesos de classe | Standard: w = N / (3·N_i) | Suavizados: raiz de N / (3·N_i), média 1 |
| Agregação por doente | Média de todas as janelas | Top-K (K = 5) |
| Decisão | Argmax de p × [5, 3, 1] | Argmax de p (sem pesos) |
| Loss / otimizador | Cross-entropy / Adam (1e-3) | Cross-entropy / Adam (1e-3) |
| Callbacks | ReduceLROnPlateau + EarlyStopping (val_wa) | ReduceLROnPlateau + EarlyStopping (val_wa) |
| **WA_murmur (split único)** | 0,6778 | 0,8296 |
| **WA_murmur (k-fold, k = 5)** | 0.6747 ±  0.0904 | 0,7715 ± 0,0451 |
